# Public Level 2 EV Charging Location Planning in NYC — Gurobi Implementation

**IEOR 4004 · Team 34 · Milestone 4**


**Contents**
1. Setup
2. Scenario parameters
3. Data preparation (sets and parameters)
4. Mathematical formulation
5. Gurobi model
6. Solution: Stage 1 and Stage 2
7. Results
8. Solution verification
9. Benchmarks and cross-checks


In [1]:
import json
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA = "../data/"
TEAM_RESULTS = "../results/geography_comparison/"

## 2. Scenario parameters

| Parameter | Symbol | Value | Source |
|---|---|---|---|
| Capital budget | $B$ | \$3,750,000 | Milestone 3 |
| Service radius | $R$ | 0.5 mile | Milestone 3 |
| Fixed first-time site cost | $F_j$ | \$20,000 | Milestone 3 |
| Incremental cost per port | $c_j$ | \$3,127 + \$3,090 = \$6,217 | Milestone 3 (ICCT 2019) |
| Weekly energy per registered EV | — | 60 kWh | Scenario assumption |
| Public Level 2 charging share | — | 25% | Scenario assumption |
| Annual demand growth, horizon | $r$, $T$ | 5%, 5 years | Scenario assumption |
| Equity-policy weight | $\lambda$ | 1.0 | Scenario assumption |
| Port power × weekly hours × availability | — | 7.2 kW × 168 h × 0.90 | Scenario assumption |
| Maximum added ports | $U_j$ | min(20, ⌊10% × remaining spaces⌋) | Scenario assumption |

Scenario assumptions are selected before optimization; they are not estimated from the data or produced by Gurobi.

In [2]:
B = 3_750_000
R = 0.5
F_FIXED = 20_000
C_PORT = 3_127 + 3_090

ENERGY_PER_EV = 60.0
PUBLIC_L2_SHARE = 0.25
r, T = 0.05, 5
LAMBDA = 1.0
PORT_KW, WEEKLY_HOURS, AVAILABILITY = 7.2, 168, 0.90
U_MAX, U_SHARE = 20, 0.10
MATCH_RADIUS = 0.05          # AFDC station within 0.05 mile of a DOT facility = same physical site

q = PORT_KW * WEEKLY_HOURS * AVAILABILITY      # effective capacity per port, kWh/week
growth = (1 + r) ** T                          # projected-demand multiplier
print(f"q = {q:,.2f} kWh/port/week")
print(f"(1+r)^T = {growth:.6f}")

q = 1,088.64 kWh/port/week
(1+r)^T = 1.276282


## 3. Data preparation

### 3.1 Demand points $I$ and baseline demand $D_i$

Electric-vehicle registrations are reported by ZIP code (NYS DMV, 2026). Each NYC ZIP total is allocated to Census
tracts using the HUD-USPS residential-address ratio (`res_ratio`), normalized over NYC tracts. Each tract is
represented by its ACS internal point.

$$D_i = \text{EVs}_i \times 60\ \text{kWh/week} \times 25\%, \qquad \text{projected demand} = D_i(1+r)^T.$$

The ZIP-point representation (one ZCTA internal point per ZIP, as in the Milestone 3 initial case) is built as well
and used as a benchmark in Section 9.

In [3]:
def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 3958.7613 * np.arcsin(np.sqrt(h))


dmv = pd.read_csv(DATA + "dmv_electric_zip_summary_2026.csv", dtype={"zip": str})
nyc_zip = dmv[(dmv["nyc_zip_lookup"] == True) & (dmv["electric_registration_records"] > 0)]

# --- Census-tract demand points (base case) ---
hud = pd.read_csv(DATA + "hud_zip_tract_crosswalk.csv", dtype={"zip": str, "tract": str})
acs = pd.read_csv(DATA + "nyc_acs_vehicle_households_2020_2024.csv", dtype={"GEOID": str})
cw = hud[hud["tract"].isin(set(acs["GEOID"])) & (hud["res_ratio"] > 0)].copy()
cw["w"] = cw["res_ratio"] / cw.groupby("zip")["res_ratio"].transform("sum")

alloc = nyc_zip[["zip", "electric_registration_records"]].merge(cw[["zip", "tract", "w"]], on="zip")
alloc["evs"] = alloc["electric_registration_records"] * alloc["w"]
tracts = (alloc.groupby("tract", as_index=False)["evs"].sum()
               .merge(acs[["GEOID", "INTPTLAT", "INTPTLONG"]], left_on="tract", right_on="GEOID"))
dem_tract = pd.DataFrame({"i": "TR:" + tracts["tract"], "lat": tracts["INTPTLAT"], "lon": tracts["INTPTLONG"],
                          "evs": tracts["evs"]})
dem_tract = dem_tract[dem_tract["evs"] > 0].reset_index(drop=True)
dem_tract["D"] = dem_tract["evs"] * ENERGY_PER_EV * PUBLIC_L2_SHARE

# --- ZIP demand points (benchmark) ---
z = nyc_zip[nyc_zip["zcta_latitude"].notna()]
dem_zip = pd.DataFrame({"i": "ZIP:" + z["zip"], "lat": z["zcta_latitude"], "lon": z["zcta_longitude"],
                        "evs": z["electric_registration_records"]}).reset_index(drop=True)
dem_zip["D"] = dem_zip["evs"] * ENERGY_PER_EV * PUBLIC_L2_SHARE

unmatched = sorted(set(nyc_zip["zip"]) - set(cw["zip"]))
print(f"NYC ZIPs with EV registrations: {len(nyc_zip)}  ({nyc_zip['electric_registration_records'].sum():,} EVs)")
print(f"Tract demand points |I|: {len(dem_tract):,}  ({dem_tract['evs'].sum():,.2f} EVs allocated)")
print(f"  ZIPs with no NYC tract match (excluded): {len(unmatched)}  "
      f"({nyc_zip[nyc_zip['zip'].isin(unmatched)]['electric_registration_records'].sum()} EVs)")
print(f"ZIP demand points (benchmark): {len(dem_zip)}")
print(f"Baseline demand  sum D_i       = {dem_tract['D'].sum():,.0f} kWh/week")
print(f"Projected demand sum D_i(1+r)^T = {dem_tract['D'].sum() * growth:,.0f} kWh/week")

NYC ZIPs with EV registrations: 215  (51,356 EVs)
Tract demand points |I|: 2,285  (51,322.00 EVs allocated)
  ZIPs with no NYC tract match (excluded): 22  (34 EVs)
ZIP demand points (benchmark): 193
Baseline demand  sum D_i       = 769,830 kWh/week
Projected demand sum D_i(1+r)^T = 982,520 kWh/week


### 3.2 Municipal candidate sites $M$

Source: NYC DOT municipal parking facilities (39 facilities) with geocoded coordinates.

**Existing ports $n^0_j$.** Taken as the larger of (i) the municipal record — the explicit L2 count, or the EV-space
count where an L2 rate is listed — and (ii) AFDC public L2 ports located within 0.05 mile of the facility.
AFDC stations matched to a facility are merged into it, so each physical charging resource appears only once in $J$.

**Data corrections applied.**
- AFDC station 261745 ("QBO – Queens Borough Hall Municipal Parking Garage", 31 ports) is assigned to the Borough Hall
  garage. Borough Hall garage and Queensboro Hall field share the address 80-25 126th Street; the DOT usage data show
  the charger was renamed from "Queensboro Hall" (through 2024-02) to "QBO – Queens Borough Hall" (from 2024-03), i.e.
  one physical site. Without this override the same 31 ports would be counted at both facilities.
- Both AFDC records for White Plains Road (4 and 18 ports, same address) are merged into that facility.
- Jerome–Gun Hill Road garage is excluded from investment eligibility: public closure announced from 2026-12-01.
  Its existing ports remain in the supply network as an other site.

**First-time indicator** $a_j = 1$ if $n^0_j = 0$. **Installation limit** $U_j = \min(20, \lfloor 0.10\,(\text{spaces}_j - n^0_j)\rfloor)$.

In [4]:
AFDC_MATCH_OVERRIDES = {261745: "borough-municipal"}
INELIGIBLE = {"gunhill": "public closure announced from 2026-12-01"}

fac = pd.read_excel(DATA + "NYC_Municipal_Parking_Locations.xlsx", header=6)
fac = fac[fac["Facility ID"].notna()]
geo = pd.read_csv(DATA + "municipal_facilities_geocoded.csv")
fac = fac.merge(geo[["facility_id", "latitude", "longitude"]], left_on="Facility ID", right_on="facility_id")
fac = fac.rename(columns={"Facility ID": "fid", "Facility name": "name", "Borough": "borough",
                          "Parking spaces": "spaces", "L2 spaces (explicit)": "l2_explicit",
                          "EV spaces (type unspecified)": "ev_unspecified", "L2 evidence": "l2_evidence",
                          "latitude": "lat", "longitude": "lon"})

def municipal_record_ports(row):
    if pd.notna(row["l2_explicit"]):
        return row["l2_explicit"]
    if str(row["l2_evidence"]).startswith("L2 rate listed") and pd.notna(row["ev_unspecified"]):
        return row["ev_unspecified"]
    return np.nan

fac["n0_record"] = fac.apply(municipal_record_ports, axis=1)

# Match AFDC public L2 stations to DOT facilities
afdc = pd.read_csv(DATA + "nyc_public_operational_l2.csv").rename(
    columns={"latitude": "lat", "longitude": "lon", "ev_level2_evse_num": "ports"})
d_af = haversine_miles(afdc["lat"].values[:, None], afdc["lon"].values[:, None],
                       fac["lat"].values[None, :], fac["lon"].values[None, :])
afdc["facility"] = np.where(d_af.min(axis=1) <= MATCH_RADIUS, fac["fid"].values[d_af.argmin(axis=1)], None)
for station_id, fid in AFDC_MATCH_OVERRIDES.items():
    afdc.loc[afdc["id"] == station_id, "facility"] = fid
fac["n0_afdc"] = fac["fid"].map(afdc.groupby("facility")["ports"].sum()).fillna(0)

fac["n0"] = np.fmax(fac["n0_record"].fillna(0), fac["n0_afdc"]).astype(int)
fac["inventory_reported"] = fac["n0_record"].notna() | (fac["n0_afdc"] > 0)
fac["a"] = (fac["n0"] == 0).astype(int)
fac["U"] = np.minimum(U_MAX, np.floor(U_SHARE * (fac["spaces"] - fac["n0"]))).clip(lower=0).astype(int)
fac["eligible"] = ~fac["fid"].isin(INELIGIBLE)
fac["j"] = np.where(fac["eligible"], "M:", "O:") + fac["fid"]

mun = fac[fac["eligible"]].reset_index(drop=True)
print(f"DOT facilities: {len(fac)};  eligible candidates |M| = {len(mun)};  excluded: {list(INELIGIBLE)}")
print(f"AFDC stations merged into DOT facilities: {afdc['facility'].notna().sum()}")
print(f"M_E (n0 > 0): {(mun['n0'] > 0).sum()}    M_N (n0 = 0): {(mun['n0'] == 0).sum()}")
print()
print(mun[["fid", "borough", "spaces", "n0_record", "n0_afdc", "n0", "a", "U"]].to_string(index=False))

DOT facilities: 39;  eligible candidates |M| = 38;  excluded: ['gunhill']
AFDC stations merged into DOT facilities: 12
M_E (n0 > 0): 13    M_N (n0 = 0): 25

                   fid       borough  spaces  n0_record  n0_afdc  n0  a  U
               belmont     The Bronx      57        NaN      0.0   0  1  5
             bronxdale     The Bronx      78        NaN      0.0   0  1  7
             jerome190     The Bronx     416        5.0      5.0   5  0 20
                 white     The Bronx      89        4.0     22.0  22  0  6
                avenue      Brooklyn      51        NaN      0.0   0  1  5
                   bay      Brooklyn     205        5.0      5.0   5  0 20
           bensonhurst      Brooklyn      92        4.0      4.0   4  0  8
          bensonhurst2      Brooklyn      24        NaN      0.0   0  1  2
              brighton      Brooklyn     312        NaN      0.0   0  1 20
              canarsie      Brooklyn     281        NaN      0.0   0  1 20
               gow

### 3.3 Other existing public sites $O$ and the supply set $J = M \cup O$

$O$ contains the AFDC public operational Level 2 stations that are not merged into an eligible DOT candidate
(plus any ineligible DOT facility with existing ports). No investment is allowed at $O$.

Existing capacity: $C^0_j = n^0_j \cdot q$.

In [5]:
other = afdc[afdc["facility"].isna()]
sites = pd.concat([
    pd.DataFrame({"j": mun["j"], "type": "M", "lat": mun["lat"], "lon": mun["lon"], "n0": mun["n0"]}),
    pd.DataFrame({"j": fac.loc[~fac["eligible"], "j"], "type": "O", "lat": fac.loc[~fac["eligible"], "lat"],
                  "lon": fac.loc[~fac["eligible"], "lon"], "n0": fac.loc[~fac["eligible"], "n0"]}),
    pd.DataFrame({"j": "O:" + other["id"].astype(str), "type": "O", "lat": other["lat"], "lon": other["lon"],
                  "n0": other["ports"]}),
], ignore_index=True)
sites["C0"] = sites["n0"] * q

assert sites["j"].is_unique
print(f"|M| = {(sites['type'] == 'M').sum()},  |O| = {(sites['type'] == 'O').sum()},  |J| = {len(sites)}")
print(f"Existing ports in J: {sites['n0'].sum():,.0f}  = AFDC {afdc['ports'].sum():,.0f}"
      f" + {sites['n0'].sum() - afdc['ports'].sum():,.0f} reported only in DOT municipal records")
print(f"Existing capacity sum C0_j: {sites['C0'].sum():,.0f} kWh/week")

|M| = 38,  |O| = 747,  |J| = 785
Existing ports in J: 3,530  = AFDC 3,508 + 22 reported only in DOT municipal records
Existing capacity sum C0_j: 3,842,899 kWh/week


### 3.4 Accessible pairs $A$ and the equity score $g_i$

$$A = \{(i,j) : d_{ij} \le R\}, \qquad
S^0_i = \sum_{j:(i,j)\in A} C^0_j, \qquad
g_i = \begin{cases}\max\left(0,\ 1 - S^0_i / D_i\right) & D_i > 0\\ 0 & D_i = 0\end{cases}$$

$S^0_i$ is an accessibility proxy: a site can be accessible from several demand points, so it is not capacity reserved
for $i$. The capacity constraints (4)–(5) prevent the same physical capacity from being allocated twice.
$g_i$ uses baseline demand $D_i$ and is held fixed across growth scenarios.

In [6]:
def build_pairs(dem, sites, radius=R):
    d = haversine_miles(dem["lat"].values[:, None], dem["lon"].values[:, None],
                        sites["lat"].values[None, :], sites["lon"].values[None, :])
    ii, jj = np.nonzero(d <= radius)
    pairs = pd.DataFrame({"i": dem["i"].values[ii], "j": sites["j"].values[jj], "dist": d[ii, jj]})
    dem = dem.copy()
    S0 = pairs.merge(sites[["j", "C0"]], on="j").groupby("i")["C0"].sum()
    dem["S0"] = dem["i"].map(S0).fillna(0.0)
    dem["g"] = np.where(dem["D"] > 0, np.maximum(0.0, 1 - dem["S0"] / dem["D"]), 0.0)
    dem["Dproj"] = dem["D"] * growth
    return dem, pairs

dem, pairs = build_pairs(dem_tract, sites)
reachable = dem["i"].isin(pairs["i"])
print(f"|A| = {len(pairs):,} accessible demand-site pairs")
print(f"Demand points with at least one site within {R} mile: {reachable.sum():,} of {len(dem):,}")
print(f"Projected demand with no accessible site: {dem.loc[~reachable, 'Dproj'].sum() / dem['Dproj'].sum():.1%}"
      f"  -> upper bound on service rate: {dem.loc[reachable, 'Dproj'].sum() / dem['Dproj'].sum():.1%}")
print(f"g_i: mean {dem['g'].mean():.3f};  share of points with g_i = 1: {(dem['g'] == 1).mean():.1%}")

|A| = 7,177 accessible demand-site pairs
Demand points with at least one site within 0.5 mile: 1,215 of 2,285
Projected demand with no accessible site: 44.8%  -> upper bound on service rate: 55.2%
g_i: mean 0.490;  share of points with g_i = 1: 49.0%


## 4. Mathematical formulation

**Sets.** $I$ demand points; $M$ eligible DOT municipal facilities; $O$ other existing public L2 sites;
$J = M \cup O$; $A \subseteq I \times J$ accessible pairs.

**Decision variables.**

| Variable | Domain | Meaning |
|---|---|---|
| $y_j$ | $\{0,1\}$, $j \in M$ | 1 if additional L2 ports are installed at facility $j$ |
| $x_j$ | $\mathbb{Z}_{\ge 0}$, $j \in M$ | number of additional L2 ports installed at facility $j$ |
| $f_{ij}$ | $\mathbb{R}_{\ge 0}$, $(i,j) \in A$ | weekly charging energy of demand point $i$ served by site $j$ (kWh/week) |

### Objective (Stage 1)

$$\max \sum_{(i,j)\in A} (1 + \lambda g_i)\, f_{ij}
\;=\; \underbrace{\sum_{(i,j)\in A} f_{ij}}_{\text{total energy served}}
\;+\; \lambda \underbrace{\sum_{(i,j)\in A} g_i f_{ij}}_{\text{equity term}}$$

Maximizes equity-weighted weekly energy served. Every kWh counts at least once; a kWh delivered to a demand point with
a larger baseline supply shortage $g_i$ receives an extra weight $\lambda g_i$. With $\lambda = 0$ the model reduces to
maximizing total energy served.

### Constraints

**(1) Capital budget**
$$\sum_{j\in M} \left(F_j a_j y_j + c_j x_j\right) \le B$$
Each added port costs $c_j$; the fixed cost $F_j$ is incurred only when a facility without existing L2 ports
($a_j = 1$) is selected.

**(2) Investment activation and installation limit**
$$y_j \le x_j \le U_j\, y_j \qquad \forall j \in M$$
If $y_j = 0$ then $x_j = 0$; if $y_j = 1$ then between 1 and $U_j$ ports are added.

**(3) Projected demand limit**
$$\sum_{j:(i,j)\in A} f_{ij} \le D_i (1+r)^T \qquad \forall i \in I$$
A demand point cannot receive more energy than its projected demand. Unmet demand is allowed; demand may be split
across several accessible sites.

**(4) Capacity at municipal facilities**
$$\sum_{i:(i,j)\in A} f_{ij} \le C^0_j + q_j x_j \qquad \forall j \in M$$
Energy served at a DOT facility is limited by its existing capacity plus $q_j$ per added port.

**(5) Capacity at other existing sites**
$$\sum_{i:(i,j)\in A} f_{ij} \le C^0_j \qquad \forall j \in O$$
Other public sites serve demand only within their existing capacity; no investment is allowed there.

**(6) Accessibility and domains**
$$f_{ij} \ge 0 \;\; \forall (i,j)\in A, \qquad x_j \in \mathbb{Z}_{\ge 0},\;\; y_j \in \{0,1\} \;\; \forall j \in M$$
Flow variables exist only for accessible pairs, so demand can be served only by sites within $R$.

### Stage 2: cost tie-break (lexicographic optimization)

The Stage 1 objective contains no cost term. Because the budget is not binding, many investment plans attain the same
Stage 1 optimum $Z^*$, including plans with ports that serve no demand. Stage 2 selects the least-cost plan among
Stage 1 optimal solutions:

$$\min \sum_{j\in M} \left(F_j a_j y_j + c_j x_j\right)
\quad \text{s.t. (1)–(6) and} \quad \sum_{(i,j)\in A}(1+\lambda g_i) f_{ij} \ge Z^* - \varepsilon .$$

Stage 2 does not change the Milestone 3 objective; it only removes unnecessary investment among equally good solutions.

## 5. Gurobi model

In [7]:
def build_model(dem, sites, mun, pairs, lam=LAMBDA, budget=B, log=False):
    m = gp.Model("nyc_public_l2_charging")
    m.Params.OutputFlag = 1 if log else 0
    m.Params.MIPGap = 0.0

    # ---- index sets ----
    I = list(dem["i"])
    M = list(mun["j"])
    O = list(sites.loc[sites["type"] == "O", "j"])
    A = list(zip(pairs["i"], pairs["j"]))

    # ---- parameters ----
    a   = dict(zip(mun["j"], mun["a"]))
    U   = dict(zip(mun["j"], mun["U"]))
    C0  = dict(zip(sites["j"], sites["C0"]))
    Dp  = dict(zip(dem["i"], dem["Dproj"]))
    g   = dict(zip(dem["i"], dem["g"]))
    F, c, q_j = F_FIXED, C_PORT, q

    # ---- decision variables ----
    y = m.addVars(M, vtype=GRB.BINARY, name="y")
    x = m.addVars(M, vtype=GRB.INTEGER, lb=0, name="x")
    f = m.addVars(A, vtype=GRB.CONTINUOUS, lb=0, name="f")            # (6): f defined only on A

    # ---- objective: equity-weighted energy served ----
    primary = gp.quicksum((1 + lam * g[i]) * f[i, j] for (i, j) in A)
    m.setObjective(primary, GRB.MAXIMIZE)

    # ---- (1) capital budget ----
    cost = gp.quicksum(F * a[j] * y[j] + c * x[j] for j in M)
    m.addConstr(cost <= budget, name="budget")

    # ---- (2) activation and installation limit ----
    m.addConstrs((y[j] <= x[j] for j in M), name="activation_lb")
    m.addConstrs((x[j] <= U[j] * y[j] for j in M), name="activation_ub")

    # ---- (3) projected demand ----
    m.addConstrs((f.sum(i, "*") <= Dp[i] for i in I), name="demand")

    # ---- (4) municipal capacity ----
    m.addConstrs((f.sum("*", j) <= C0[j] + q_j * x[j] for j in M), name="capacity_municipal")

    # ---- (5) other-site capacity ----
    m.addConstrs((f.sum("*", j) <= C0[j] for j in O), name="capacity_other")

    m._vars = {"y": y, "x": x, "f": f}
    m._expr = {"primary": primary, "cost": cost}
    m._sets = {"I": I, "M": M, "O": O, "A": A}
    return m


def solve_stage2(m, log=False):
    # Stage 2: hold the Stage 1 optimum and minimize investment cost
    z_star = m.ObjVal
    eps = max(1e-4, 1e-7 * abs(z_star))
    m.addConstr(m._expr["primary"] >= z_star - eps, name="hold_stage1_optimum")
    m.setObjective(m._expr["cost"], GRB.MINIMIZE)
    m.Params.OutputFlag = 1 if log else 0
    m.optimize()
    return z_star


def summarize(m, dem, pairs, label, z_star=None):
    y, x, f = m._vars["y"], m._vars["x"], m._vars["f"]
    M, A = m._sets["M"], m._sets["A"]
    flow = np.array([f[k].X for k in A])
    positive = flow > 1e-6
    return {
        "case": label,
        "status": "OPTIMAL" if m.Status == GRB.OPTIMAL else str(m.Status),
        "MIP gap": m.MIPGap,
        "stage-1 objective": z_star if z_star is not None else m._expr["primary"].getValue(),
        "selected sites": int(round(sum(y[j].X for j in M))),
        "added ports": int(round(sum(x[j].X for j in M))),
        "cost ($)": m._expr["cost"].getValue(),
        "served (kWh/wk)": flow.sum(),
        "service rate": flow.sum() / dem["Dproj"].sum(),
        "max flow distance (mi)": pairs["dist"].values[positive].max() if positive.any() else 0.0,
        "selected": sorted(j[2:] for j in M if y[j].X > 0.5),
    }


def table(rows):
    t = pd.DataFrame(rows).drop(columns="selected").set_index("case")
    return t.to_string(formatters={
        "MIP gap": "{:.4%}".format, "stage-1 objective": "{:,.2f}".format, "cost ($)": "{:,.0f}".format,
        "served (kWh/wk)": "{:,.2f}".format, "service rate": "{:.2%}".format,
        "max flow distance (mi)": "{:.4f}".format})


m = build_model(dem, sites, mun, pairs)
m.update()
print(f"Variables: {m.NumVars:,}  (binary {m.NumBinVars}, integer incl. binary {m.NumIntVars}, "
      f"continuous {m.NumVars - m.NumIntVars:,})")
print(f"Constraints: {m.NumConstrs:,}")

Set parameter WLSAccessID
Set parameter WLSSecret
Variables: 7,253  (binary 38, integer incl. binary 76, continuous 7,177)
Constraints: 3,147


## 6. Solution

### 6.1 Stage 1 — Milestone 3 objective only

In [8]:
m.Params.OutputFlag = 1
m.optimize()
stage1 = summarize(m, dem, pairs, "Stage 1 only")
x1 = {j: round(m._vars["x"][j].X) for j in m._sets["M"]}
f1 = pd.Series({k: m._vars["f"][k].X for k in m._sets["A"]})

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0

Optimize a model with 3147 rows, 7253 columns and 14607 nonzeros (Max)
Model fingerprint: 0x6c1267f1
Model has 7177 linear objective coefficients
Variable types: 7177 continuous, 76 integer (38 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e-02, 4e+06]

Found heuristic solution: objective 461933.79012
Presolve removed 3147 rows and 7253 columns
Presolve time: 0.01s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 1 (of 8 available processors)

Solution count 3: 519140 519140 461934 

Optimal solution found (tolerance 0.00e+00)
Best objective 5.19139

Stage 1 alone does not determine the investment plan uniquely. The table compares the ports Gurobi installed with the
ports actually required by the energy routed to each facility.

In [9]:
served_at = f1.groupby(level=1).sum()
check = pd.DataFrame({
    "ports installed": pd.Series(x1),
    "ports required": [int(np.ceil(max(0.0, served_at.get(j, 0.0) - C0) / q - 1e-9))
                       for j, C0 in zip(mun["j"], mun["n0"] * q)],
}, index=mun["j"]).loc[lambda t: t["ports installed"] > 0]
check.index = check.index.str[2:]
print(check.to_string())
print(f"\nTotal ports installed: {check['ports installed'].sum()}   required: {check['ports required'].sum()}")
print(f"Stage 1 investment cost: ${stage1['cost ($)']:,.0f}")

                        ports installed  ports required
j                                                      
belmont                               5               1
bronxdale                             7               0
jerome190                            20               0
white                                 6               0
avenue                                5               3
bay                                  20               0
bensonhurst                           8               0
bensonhurst2                          2               2
brighton                             20               2
canarsie                             20               2
gowanus                              20               0
grant                                 5               5
meeker                               20               0
sheepshead                            4               0
sheepshead2                           7               0
west-brighton                         4         

### 6.2 Stage 2 — least-cost plan among Stage 1 optimal solutions

In [10]:
z_star = solve_stage2(m, log=True)
base = summarize(m, dem, pairs, "Base case (Stage 1 + Stage 2)", z_star=z_star)

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0

Optimize a model with 3148 rows, 7253 columns and 21784 nonzeros (Min)
Model fingerprint: 0xc6316f50
Model has 63 linear objective coefficients
Variable types: 7177 continuous, 76 integer (38 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [6e+03, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e-02, 4e+06]

MIP start from previous solve produced solution with objective 3.08627e+06 (0.01s)
Loaded MIP start from previous solve with objective 3.08627e+06

Presolve removed 3148 rows and 7253 columns
Presolve time: 0.01s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.03 work units)
Thread count was 1 (of 8 available processors)

Solution c

## 7. Results

In [11]:
print(table([stage1, base]))
print(f"\nBudget slack: ${B - base['cost ($)']:,.0f}")

                                status MIP gap stage-1 objective  selected sites  added ports  cost ($) served (kWh/wk) service rate max flow distance (mi)
case                                                                                                                                                       
Stage 1 only                   OPTIMAL 0.0000%        519,139.57              38          416 3,086,272      493,415.14       50.22%                 0.4998
Base case (Stage 1 + Stage 2)  OPTIMAL 0.0000%        519,139.57              14           31   472,727      493,415.12       50.22%                 0.4998

Budget slack: $3,277,273


In [12]:
y, x, f = m._vars["y"], m._vars["x"], m._vars["f"]
flow = pd.Series({k: f[k].X for k in m._sets["A"]})
served_site = flow.groupby(level=1).sum()

plan = mun[["j", "fid", "name", "borough", "spaces", "n0", "a", "U"]].copy()
plan["added ports"] = plan["j"].map(lambda j: round(x[j].X))
plan = plan[plan["added ports"] > 0].copy()
plan["cost ($)"] = F_FIXED * plan["a"] + C_PORT * plan["added ports"]
plan["capacity (kWh/wk)"] = (plan["n0"] + plan["added ports"]) * q
plan["served (kWh/wk)"] = plan["j"].map(served_site).fillna(0.0)
plan["utilization"] = plan["served (kWh/wk)"] / plan["capacity (kWh/wk)"]
plan = plan.sort_values("added ports", ascending=False).drop(columns="j")
print(plan.to_string(index=False, formatters={"cost ($)": "{:,.0f}".format,
      "capacity (kWh/wk)": "{:,.1f}".format, "served (kWh/wk)": "{:,.1f}".format, "utilization": "{:.1%}".format}))
print()
print(plan.groupby("borough")[["added ports", "cost ($)"]].sum().to_string())

          fid                                      name       borough  spaces  n0  a  U  added ports cost ($) capacity (kWh/wk) served (kWh/wk) utilization
        grant Grant Avenue West Municipal Parking Field      Brooklyn      55   0  1  5            5   51,085           5,443.2         5,443.2      100.0%
      bayside           Bayside Municipal Parking Field        Queens      85   0  1  8            4   44,868           4,354.6         3,503.2       80.4%
       avenue          Avenue M Municipal Parking Field      Brooklyn      51   0  1  5            3   38,651           3,265.9         2,301.1       70.5%
queensvillage    Queens Village Municipal Parking Field        Queens      52   0  1  5            3   38,651           3,265.9         2,437.8       74.6%
      newDorp          New Dorp Municipal Parking Field Staten Island      75   0  1  7            3   38,651           3,265.9         2,195.4       67.2%
 bensonhurst2    Bensonhurst #2 Municipal Parking Field      Bro

## 8. Solution verification

Each constraint is re-evaluated from the returned variable values, independently of Gurobi's own feasibility tolerances.

In [13]:
tol = 1e-6
served_i = flow.groupby(level=0).sum().reindex(dem["i"]).fillna(0.0)
served_j = flow.groupby(level=1).sum().reindex(sites["j"]).fillna(0.0)
x_val = pd.Series({j: x[j].X for j in mun["j"]})
y_val = pd.Series({j: y[j].X for j in mun["j"]})
cap = sites.set_index("j")["C0"] + x_val.reindex(sites["j"]).fillna(0.0).values * q
spent = (F_FIXED * mun.set_index("j")["a"] * y_val + C_PORT * x_val).sum()
U_val = mun.set_index("j")["U"]
pos_pairs = pairs[flow.reindex(list(zip(pairs["i"], pairs["j"]))).values > tol]

checks = {
    "(1) budget: cost <= B": spent <= B + tol,
    "(2) activation: y <= x <= U y": bool(((y_val <= x_val + tol) & (x_val <= U_val * y_val + tol)).all()),
    "(2) integrality of x and y": bool(np.allclose(x_val, x_val.round()) and np.allclose(y_val, y_val.round())),
    "(3) demand: served_i <= D_i(1+r)^T": bool((served_i.values <= dem["Dproj"].values + tol).all()),
    "(4)-(5) capacity: served_j <= C0_j + q x_j": bool((served_j.values <= cap.values + tol).all()),
    "(6) nonnegativity of f": bool((flow >= -tol).all()),
    "(6) accessibility: positive flow only within R": bool((pos_pairs["dist"] <= R + 1e-9).all()),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\nObjective recomputed from f: {((1 + LAMBDA * dem.set_index('i')['g'].reindex(flow.index.get_level_values(0)).values) * flow.values).sum():,.2f}"
      f"   (Gurobi Z* = {z_star:,.2f})")
print(f"Max distance among positive flows: {pos_pairs['dist'].max():.4f} mile")

PASS  (1) budget: cost <= B
PASS  (2) activation: y <= x <= U y
PASS  (2) integrality of x and y
PASS  (3) demand: served_i <= D_i(1+r)^T
PASS  (4)-(5) capacity: served_j <= C0_j + q x_j
PASS  (6) nonnegativity of f
PASS  (6) accessibility: positive flow only within R

Objective recomputed from f: 519,139.52   (Gurobi Z* = 519,139.57)
Max distance among positive flows: 0.4998 mile


**Existing-inventory evidence for selected facilities.** Facilities with no reported L2 inventory are modeled with
$n^0_j = 0$. As supporting evidence, the selected facilities are checked against the NYC DOT municipal EV charging
session data (`municipal_usage_monthly.csv`), which records sessions at DOT-operated municipal chargers.

In [14]:
usage = pd.read_csv(DATA + "municipal_usage_monthly.csv")
usage_names = usage["location_name"].str.lower()
for _, s in plan.iterrows():
    key = s["name"].replace(" Municipal Parking Field", "").replace(" Municipal Parking Garage", "").lower()
    hits = usage_names.str.contains(key, regex=False).sum()
    print(f"{s['fid']:<16} reported inventory: {'yes' if fac.set_index('fid').at[s['fid'], 'inventory_reported'] else 'no ':<4}"
          f"  DOT charging-session records: {hits}")

grant            reported inventory: no    DOT charging-session records: 0
bayside          reported inventory: no    DOT charging-session records: 0
avenue           reported inventory: no    DOT charging-session records: 0
queensvillage    reported inventory: no    DOT charging-session records: 0
newDorp          reported inventory: no    DOT charging-session records: 0
bensonhurst2     reported inventory: no    DOT charging-session records: 0
brighton         reported inventory: no    DOT charging-session records: 0
canarsie         reported inventory: no    DOT charging-session records: 0
greatkills       reported inventory: no    DOT charging-session records: 0
belmont          reported inventory: no    DOT charging-session records: 0
ditmars2         reported inventory: no    DOT charging-session records: 0
flushing4        reported inventory: no    DOT charging-session records: 0
rockaway         reported inventory: no    DOT charging-session records: 0
rosedale         reported

## 9. Benchmarks and cross-checks

### 9.1 Unweighted benchmark ($\lambda = 0$)

In [15]:
def solve_case(dem_points, label, lam=LAMBDA):
    d, p = build_pairs(dem_points, sites)
    mm = build_model(d, sites, mun, p, lam=lam)
    mm.optimize()
    zs = solve_stage2(mm)
    return summarize(mm, d, p, label, z_star=zs)

lam0 = solve_case(dem_tract, "lambda = 0", lam=0.0)
print(table([base, lam0]))
print("\nSame selected facilities:", base["selected"] == lam0["selected"])

                                status MIP gap stage-1 objective  selected sites  added ports cost ($) served (kWh/wk) service rate max flow distance (mi)
case                                                                                                                                                      
Base case (Stage 1 + Stage 2)  OPTIMAL 0.0000%        519,139.57              14           31  472,727      493,415.12       50.22%                 0.4998
lambda = 0                     OPTIMAL 0.0000%        493,415.14              14           31  472,727      493,415.09       50.22%                 0.4998

Same selected facilities: True


### 9.2 ZIP-point demand representation (Milestone 3 initial case)

In [16]:
zip_case = solve_case(dem_zip, "ZIP points")
print(table([base, zip_case]))

                                status MIP gap stage-1 objective  selected sites  added ports cost ($) served (kWh/wk) service rate max flow distance (mi)
case                                                                                                                                                      
Base case (Stage 1 + Stage 2)  OPTIMAL 0.0000%        519,139.57              14           31  472,727      493,415.12       50.22%                 0.4998
ZIP points                     OPTIMAL 0.0000%        477,256.53               9           44  453,548      418,368.39       42.58%                 0.4957


### 9.3 Cross-check against the committed team results

This notebook and the repository pipeline (`src/`) were developed independently. Outcomes are compared below.
Parameter-level differences are expected only where Section 3.2 applies a data correction
(Queens Borough Hall override, White Plains Road merge, Gun Hill eligibility).

In [17]:
rows = []
for label, mine in [("tract", base), ("zip", zip_case)]:
    s = json.load(open(TEAM_RESULTS + f"{label}/summary.json"))
    sel = pd.read_csv(TEAM_RESULTS + f"{label}/municipal_candidate_solution.csv")
    rows.append({
        "case": label,
        "sites (mine/team)": f"{mine['selected sites']} / {s['selected_municipal_sites']}",
        "ports (mine/team)": f"{mine['added ports']} / {s['additional_ports']}",
        "cost (mine/team)": f"{mine['cost ($)']:,.0f} / {s['investment_cost_usd']:,.0f}",
        "served (mine/team)": f"{mine['served (kWh/wk)']:,.2f} / {s['total_served_kwh']:,.2f}",
        "same facilities": mine["selected"] == sorted(sel.loc[sel["selected"] == 1, "facility_id"]),
    })
print(pd.DataFrame(rows).set_index("case").to_string())

      sites (mine/team) ports (mine/team)   cost (mine/team)       served (mine/team)  same facilities
case                                                                                                  
tract           14 / 14           31 / 31  472,727 / 472,727  493,415.12 / 493,415.12             True
zip               9 / 9           44 / 44  453,548 / 453,548  418,368.39 / 418,368.39             True
